In [22]:
#!/usr/bin/env python3
from datetime import datetime
import gymnasium as gym
import rospy 
import cv2
from cv_bridge import CvBridge, CvBridgeError
from sensor_msgs.msg import Image,Imu,LaserScan
from geometry_msgs.msg import Twist,Vector3
from nav_msgs.msg import Odometry
import queue
from gymnasium.spaces import Dict,Box
import numpy as np
from queue import Queue
import time
from jaxrl2.data import VariableCapacityBuffer,ReplayBuffer
import message_filters
from scipy.spatial.transform import Rotation



In [23]:
rospy.init_node("DATA_FILTER", anonymous=False)

In [24]:
IMAGE_TOPIC="/camera/image_raw"
IMU_TOPIC="/imu"
WHEEL_ODOMETRY_TOPIC="/wheel_odom_with_covariance"
ROBOT_CMD_TOPIC="/cmd_vel"
# LIDAR_TOPIC="/scan"
RATE = 60
DT=1/RATE

In [25]:
def ros_vector3_to_np_array(msg):
    return np.array([msg.x,-1*msg.y,msg.z])

In [26]:
bridge=CvBridge()
image_size=64
hold_action_queue=True
# Define Queue
image_queue=Queue()
imu_queue=Queue()
wheel_queue=Queue()
action_queue=Queue()


#
image_space = Box(
            low=0,
            high=255,
            shape=(image_size, image_size, 3,1),
            dtype=np.uint8,
        )
        
vec_space = Box(
    low=-5.1,
    high=5.1,
    shape=(4,),
    dtype=np.float32,
)

observation_space=Dict({"pixels":image_space, "vector":vec_space})

action_space = Box(
    low=np.array([-1.0, -1.0]),  # [steering, throttle/brake]
    high=np.array([1.0, 1.0]),
    dtype=np.float32
)
def image_callback(image):
    # print("image")
    image = bridge.imgmsg_to_cv2(image, "rgb8")
    image=cv2.resize(image,(image_size,image_size))
    
    image_queue.put(image)

def imu_callback(imu):
    orientation=imu.orientation
    # payload=dict(w=ros_vector3_to_np_array(imu.angular_velocity),a=ros_vector3_to_np_array())
    payload = Rotation.from_quat(np.array([orientation.x,orientation.y,orientation.z,orientation.w]))
    payload = payload.as_euler('xyz', degrees=True)
    imu_queue.put(payload[2])

def wheel_callback(wheel:Odometry):
    # payload=dict(w=ros_vector3_to_np_array(wheel.angular_velocity),a=ros_vector3_to_np_array(wheel.linear_acceleration))
    # imu_queue.put(payload)
    wheel_queue.put(wheel.twist.twist.linear.x)

def action_callback(action):

    global hold_action_queue
    if not hold_action_queue:

        action_queue.put(action)
        hold_action_queue = True



In [27]:

# image_sub = rospy.Subscriber(IMAGE_TOPIC,Image,image_callback)
# lidar_sub = rospy.Subscriber(LIDAR_TOPIC,LaserScan,lidar_callback)
# imu_sub = rospy.Subscriber(IMU_TOPIC,Imu,imu_callback)



def callback(wheel,imu,image):
    global hold_action_queue
    hold_action_queue = False
    imu_callback(imu)
    image_callback(image)
    wheel_callback(wheel)
    # action_callback(action)

image_sub = message_filters.Subscriber(IMAGE_TOPIC,Image)
wheel_sub = message_filters.Subscriber(WHEEL_ODOMETRY_TOPIC,Odometry)
imu_sub = message_filters.Subscriber(IMU_TOPIC,Imu)
# robot_cmd = message_filters.Subscriber(ROBOT_CMD_TOPIC,Twist)
ts = ts = message_filters.ApproximateTimeSynchronizer([wheel_sub,imu_sub,image_sub], 15,4,True)
ts.registerCallback(callback)

robot_cmd =rospy.Subscriber(ROBOT_CMD_TOPIC, Twist, action_callback)

time.sleep(1.0)

step=0
#put everything in a list becuase we need some metrics
a=[]
w=[]
wheels=[]
images=[]
actions=[]
# rospy.spin()
while True:
    # continue
    # time.sleep(2.0)

    print("image",image_queue.qsize())
    image=image_queue.get()
    # print("imu",imu_queue.qsize())
    imu=imu_queue.get()
    # print("wheel",wheel_queue.qsize())
    wheel=wheel_queue.get()
    # print("action",action_queue.qsize())
    action=action_queue.get()

    # print("Image",image)
    a.append(wheel)
    w.append(imu)
    actions.append([action.angular.z,action.linear.x])   #steer throttle
    images.append(image)
    # action_queue.queue.clear()
    time.sleep(0.4)

    if (image_queue.empty() or imu_queue.empty() or wheel_queue.empty() or action_queue.empty()):
        break

    # more data
    # image_queue.put(image)
    # imu=imu_queue.get(imu)
    # scan=lidar_queue.put(scan)
    # step+=1




image 12
image 17
image 22
image 24
image 29
image 34
image 36
image 41
image 43
image 48
image 53
image 55
image 60
image 65
image 70
image 72
image 77
image 79
image 84
image 89
image 91
image 96
image 98
image 103
image 108
image 110
image 115
image 120
image 122
image 127
image 129
image 134
image 139
image 144
image 146
image 148
image 153
image 158
image 160
image 165
image 167
image 172
image 174
image 179
image 184
image 186
image 188
image 193
image 198
image 200
image 205
image 210
image 212
image 217
image 219
image 224
image 226
image 231
image 236
image 238
image 243
image 248
image 250
image 255
image 260
image 265
image 267
image 272
image 277
image 282
image 287
image 290
image 294
image 299
image 304
image 309
image 311
image 316
image 321
image 323
image 328
image 330
image 335
image 337
image 342
image 347
image 352
image 357
image 361
image 364
image 369
image 374
image 379
image 384
image 386
image 391
image 396
image 401
image 406
image 411
image 416
image 418
ima

In [28]:
# from collections import deque
# import numpy as np
# import cv2
# import os
# import glob
# import pickle
# # create replay buffer for behavior cloning

# def add_vector_noise(vector, noise_scale=0.05):
#     return vector + np.random.normal(0, noise_scale, size=vector.shape)

# v = np.zeros((3,))
# vs = []
# speeds = []
# # angle = np.zeros((3,))
# headings = []
# distance_to_obstacle = []
# observation = None
# p_action = np.zeros((2,))

# replay_buffer = ReplayBuffer(
#     observation_space,
#     action_space,
#     int(1e5)
# )
# done = False

# use_augmentation = True
# augmentation_probability = 0.5
# noise_scale = 0.03

# while replay_buffer._size < int(2e4):
#     target_speed = np.mean(a)
#     target_heading = np.random.uniform(1e-8, 2*np.pi)
#     collision_threshold = 0.4
#     v = np.zeros((3,))
#     for i in range(len(a)):
#         if i >= len(actions)-1:
#             break
#         v = a[i]
#         heading = w[i]
#         action = actions[1]
#         headings.append(heading)

#         vec = np.zeros(4)
#         vec[0] = p_action[0]/1.0
#         vec[1] = p_action[1]/1.0
#         vec[2] = np.clip(v/(target_speed+1.0), 0, 1.0)
#         vec[3] = np.cos(abs(heading-target_heading))
        
#         WEIGHTS = {
#             'speed': 0.6,
#             'direction': 0.2,
#             'action_smoothness': 0.05,
#             'time_penalty': 0.09,
#             'collision_penalty': 2.0,  # Still the largest penalty but scaled down
#             'lane_departure_penalty': 2.0,
#             'goal_reached_bonus': 2.0
#         }
        
#         # Missing direction_factor definition - adding it here
#         heading_diff = abs(heading - target_heading)
#         direction_factor = np.cos(heading_diff)
        
#         speed_factor = v / target_speed
#         normalized_speed = np.clip(speed_factor, 0, 1.0)
#         speed_reward = normalized_speed
        
#         # 2. Direction component: alignment with target heading
#         direction_reward = max(direction_factor, 0)  # Only reward positive alignment
        
#         # 3. Action smoothness component (steer is set elsewhere)
#         action_smoothness_reward = -abs(action[0])  # Penalize large steering actions
        
#         reward = 0.0
        
#         # Add positive components
#         if speed_factor < 1.0:
#             reward += WEIGHTS['speed'] * speed_reward
        
#         # Add direction reward only in non-exploration mode
#         # if not self.explore_mode:
#         reward += WEIGHTS['direction'] * direction_reward
        
#         # Add action smoothness
#         reward += WEIGHTS['action_smoothness'] * action_smoothness_reward
        
#         # Apply time penalty to encourage efficiency
#         reward -= WEIGHTS['time_penalty']
        
#         # Ensure reward is numerically stable
#         reward = np.nan_to_num(reward, nan=0.0)
        
#         # img = np.array(images[i]).astype(np.float32)[..., None]/255
        
#         vector_state = vec.copy()
#         next_observation = dict(
#             # pixels=img,
#             vector=vector_state
#         )
        
#         # First time through the loop, observation will be None
#         if observation is None:
#             observation = next_observation
#             continue  # Skip to the next iteration, as we can't add to buffer yet
        
#         # Check conditions for done state
#         # Missing speed definition, using v as a substitute
#         speed = v
#         # Missing dist_to_obs definition, adding placeholder
#         dist_to_obs = distance_to_obstacle[i] if i < len(distance_to_obstacle) else 1.0
        
#         done = False
        
#         if i % 200 == 0:
#             done = True
#             reward += 10
        
#         if dist_to_obs <= collision_threshold:
#             done = True
#             reward -= 10
        
#         if speed > target_speed:
#             done = True
#             reward -= 10
        
#         mask = 1-int(done)
        
#         # Insert into replay buffer - only once per iteration
#         replay_buffer.insert(
#             dict(
#                 observations=observation,
#                 actions=action,
#                 rewards=reward,
#                 masks=mask,
#                 dones=done,
#                 next_observations=next_observation,
#             )
#         )
        
#         # Update for next iteration
#         p_action = action
#         observation = next_observation
        
#         # Reset done state for next iteration
#         done = False
# dataset_folder = os.path.join("datasets")
# os.makedirs(dataset_folder, exist_ok=True)
# count=len(glob.glob(f"{dataset_folder}/*.pkl"))
# final_dataset_file = os.path.join(dataset_folder, f"rl_data_{count}.pkl")
# with open(final_dataset_file, "wb") as f:
#     pickle.dump(replay_buffer, f)

# print("Target Speed", np.mean(speeds))
# print("Target Heading", np.rad2deg(np.max(headings)))
# print("Buffer size:", replay_buffer._size)
# print("Augmentation enabled:", use_augmentation)

In [32]:
from collections import deque
import random
import numpy as np
import cv2
# create replay buffer for behavior cloning

def add_vector_noise(vector, noise_scale=0.05):
    return vector + np.random.normal(0, noise_scale, size=vector.shape)

v = np.zeros((3,))
vs = []
speeds = []
# angle = np.zeros((3,))
headings = []
distance_to_obstacle = []
observation = None
p_action = np.zeros((2,))

replay_buffer = ReplayBuffer(
    observation_space,
    action_space,
    int(1e5)
)
done = False

use_augmentation = True
augmentation_probability = 0.5
noise_scale = 0.03

# while replay_buffer._size < int(2e4):
target_speed = np.mean(a)
target_heading = np.random.uniform(1e-8, 2*np.pi)
collision_threshold = 0.4
v = np.zeros((3,))

for k in range(20):
    i=0
    if k % 2 ==0 :
        target_heading =-1
    else:
        target_heading = random.choice(w[i:])
    for i in range(len(a)):
        if i >= len(actions)-1:
            break
        v = a[i]
        heading = w[i]
        action = actions[1]
        headings.append(heading)

        vec = np.zeros(4)
        vec[0] = p_action[0]/1.0
        vec[1] = p_action[1]/1.0
        vec[2] = np.clip(v/(target_speed+1.0), 0, 1.0)
        if target_heading == -1:
            vec[3] = np.cos(np.pi)
        else:
            vec[3] = np.cos(abs(heading-target_heading))
        WEIGHTS = {
            'speed': 0.6,
            'direction': 0.2,
            'action_smoothness': 0.05,
            'time_penalty': 0.09,
            'collision_penalty': 2.0,  # Still the largest penalty but scaled down
            'lane_departure_penalty': 2.0,
            'goal_reached_bonus': 2.0
        }
        
        # Missing direction_factor definition - adding it here
        heading_diff = abs(heading - target_heading)
        direction_factor = np.cos(heading_diff)
        
        speed_factor = v / target_speed
        normalized_speed = np.clip(speed_factor, 0, 1.0)
        speed_reward = normalized_speed
        
        # 2. Direction component: alignment with target heading
        direction_reward = max(direction_factor, 0)  # Only reward positive alignment
        
        # 3. Action smoothness component (steer is set elsewhere)
        action_smoothness_reward = -abs(action[0])  # Penalize large steering actions
        
        reward = 0.0
        
        # Add positive components
        if speed_factor < 1.0:
            reward += WEIGHTS['speed'] * speed_reward
        
        # Add direction reward only in non-exploration mode
        # if not self.explore_mode:
        reward += WEIGHTS['direction'] * direction_reward
        
        # Add action smoothness
        reward += WEIGHTS['action_smoothness'] * action_smoothness_reward
        
        # Apply time penalty to encourage efficiency
        reward -= WEIGHTS['time_penalty']
        
        # Ensure reward is numerically stable
        reward = np.nan_to_num(reward, nan=0.0)
        
        img = np.array(images[i]).astype(np.float32)[..., None]/255
        
        vector_state = vec.copy()
        next_observation = dict(
            pixels=img,
            vector=vector_state
        )
        
        # First time through the loop, observation will be None
        if observation is None:
            observation = next_observation
            continue  # Skip to the next iteration, as we can't add to buffer yet
        
        # Check conditions for done state
        # Missing speed definition, using v as a substitute
        speed = v
        # Missing dist_to_obs definition, adding placeholder
        dist_to_obs = distance_to_obstacle[i] if i < len(distance_to_obstacle) else 1.0
        
        done = False
        
        if i % 200 == 0:
            done = True
            reward += 10
        
        if dist_to_obs <= collision_threshold:
            done = True
            reward -= 10
        
        if speed > target_speed:
            done = True
            reward -= 10
        
        mask = 1-int(done)
        
        # Insert into replay buffer - only once per iteration
        replay_buffer.insert(
            dict(
                observations=observation,
                actions=action,
                rewards=reward,
                masks=mask,
                dones=done,
                next_observations=next_observation,
            )
        )
        
        # Update for next iteration
        p_action = action
        observation = next_observation
        
        # Reset done state for next iteration
        done = False

# headings
import glob
import os
import pickle


dataset_folder = os.path.join("datasets")
os.makedirs(dataset_folder, exist_ok=True)
count=len(glob.glob(f"{dataset_folder}/*.pkl"))
final_dataset_file = os.path.join(dataset_folder, f"bc_data_{count}.pkl")
with open(final_dataset_file, "wb") as f:
    pickle.dump(replay_buffer, f)

print("Target Speed", np.mean(speeds))
# print("Target Heading", np.rad2deg(np.max(headings)))
print("Buffer size:", replay_buffer._size)
print("Augmentation enabled:", use_augmentation)

Target Speed nan
Buffer size: 58479
Augmentation enabled: True


In [ ]:
# replay_buffer._size

In [ ]:
# import matplotlib.pyplot as plt
# x=np.array(range(len(headings)))
# # plt.plot(x,accel_x,label="Acceleration")
# # plt.plot(x,w[:,2],label="Angular Velocity")
# # plt.plot(x,heading,label="Heading")
# # plt.plot(x,vs,label="Velocity")
# # plt.plot(x,speed,label="Speed")
# plt.plot(x,distance_to_obstacle,label="Obstacle")
# # plt.plot(x,smoothed_heading,label="Smooth Heading")
# plt.legend()